![Identidade visual da disciplina](../capa-analise-visualizacao-python-ia.png)

# IA001 — Análise e Visualização de Dados com Python e Ferramentas Assistidas por IA

## Introdução ao Streamlit

**Objetivo:** transformar análises e gráficos Python em uma interface web com filtros, menus e abas, sem escrever HTML ou JavaScript.

**Aplicação completa:** `notebooks/streamlit_demo/app.py`.

## 1. O que é Streamlit?

Streamlit é uma biblioteca para criar aplicações de dados em Python. O arquivo da aplicação é um script comum, executado de cima para baixo.

Quando o usuário altera um widget, o script é executado novamente. Variáveis, filtros e gráficos são recalculados com o novo valor.

```text
widget alterado → script executado → interface atualizada
```

## 2. Instalação e execução

No terminal, a partir da raiz do curso:

```bash
python -m pip install -r notebooks/streamlit_demo/requirements.txt
streamlit run notebooks/streamlit_demo/app.py
```

O comando inicia um servidor local e informa o endereço da aplicação, normalmente `http://localhost:8501`. Para encerrar, pressione `Ctrl+C` no terminal.

## 3. A menor aplicação possível

Salve o código abaixo em um arquivo `.py` e execute-o com `streamlit run`:

```python
import streamlit as st

st.title("Minha primeira aplicação")
nome = st.text_input("Qual é o seu nome?")
if nome:
    st.success(f"Olá, {nome}!")
```

Funções `st.*` escrevem elementos na página. Widgets também retornam valores para o programa.

## 4. Elementos fundamentais

| Finalidade | Exemplos |
|---|---|
| Texto | `st.title`, `st.header`, `st.markdown`, `st.caption` |
| Dados | `st.dataframe`, `st.metric`, `st.json` |
| Widgets | `st.selectbox`, `st.multiselect`, `st.slider`, `st.button` |
| Layout | `st.sidebar`, `st.columns`, `st.tabs`, `st.expander` |
| Gráficos | `st.altair_chart`, `st.plotly_chart`, `st.pyplot` |
| Desempenho | `@st.cache_data` para dados e cálculos repetidos |

## 5. Dados do exemplo

Vamos reutilizar uma pequena base fictícia de vendas. No aplicativo, a função que cria os dados recebe `@st.cache_data`, evitando trabalho repetido em cada interação.

In [1]:
import altair as alt
import pandas as pd

meses = pd.date_range("2025-01-01", periods=12, freq="MS")
categorias = ["Eletrônicos", "Casa", "Livros"]
regioes = ["Sul", "Sudeste"]
linhas = []

for i, mes in enumerate(meses):
    for j, categoria in enumerate(categorias):
        for k, regiao in enumerate(regioes):
            unidades = 72 + 7*i + 18*j + 12*k + ((i + 2*j) % 4)*6
            preco = [320, 140, 55][j]
            linhas.append({
                "mes": mes, "categoria": categoria, "regiao": regiao,
                "unidades": unidades, "receita": unidades * preco,
                "publicidade": 3200 + 310*i + 900*j + 450*k
            })

dados = pd.DataFrame(linhas)
dados.head()

,mes,categoria,regiao,unidades,receita,publicidade
0,2025-01-01,Eletrônicos,Sul,72,23040,3200
1,2025-01-01,Eletrônicos,Sudeste,84,26880,3650
2,2025-01-01,Casa,Sul,102,14280,4100
3,2025-01-01,Casa,Sudeste,114,15960,4550
4,2025-01-01,Livros,Sul,108,5940,5000


## 6. Incorporando um gráfico Altair

O gráfico é criado normalmente. No Streamlit, substituímos a última linha pela função `st.altair_chart`.

In [2]:
grafico = (
    alt.Chart(dados)
    .mark_bar()
    .encode(
        x=alt.X("sum(receita):Q", title="Receita (R$)"),
        y=alt.Y("categoria:N", title=None, sort="-x"),
        color=alt.Color("categoria:N", legend=None),
        tooltip=["categoria:N", alt.Tooltip("sum(receita):Q", format=",.0f")]
    )
    .properties(height=300, title="Receita por categoria")
)
grafico

alt.Chart(...)

No aplicativo, usamos:

```python
st.altair_chart(grafico, width="stretch")
```

`width="stretch"` faz o gráfico ocupar a largura disponível. Seleções e tooltips do Altair continuam funcionando.

## 7. Filtros reativos

O valor retornado pelo widget é usado para filtrar os dados antes de criar o gráfico:

```python
categorias = st.multiselect(
    "Categorias", dados["categoria"].unique(),
    default=dados["categoria"].unique()
)
filtrado = dados[dados["categoria"].isin(categorias)]
grafico = alt.Chart(filtrado)...
```

Não é necessário criar callbacks para esse fluxo básico.

## 8. Menus, colunas e abas

Um menu lateral pode escolher a seção da aplicação:

```python
with st.sidebar:
    pagina = st.radio("Seção", ["Início", "Altair", "Plotly", "Seaborn"])

if pagina == "Altair":
    aba1, aba2 = st.tabs(["Barras", "Linhas"])
    with aba1:
        st.altair_chart(grafico_barras, width="stretch")
    with aba2:
        st.altair_chart(grafico_linhas, width="stretch")
```

Para indicadores lado a lado, use `col1, col2 = st.columns(2)` e escreva em cada coluna.

## 9. Outras bibliotecas

Streamlit possui funções próprias para as bibliotecas usadas nos notebooks anteriores:

```python
# Plotly
st.plotly_chart(fig_plotly, width="stretch")

# Seaborn/Matplotlib
fig, ax = plt.subplots()
sns.boxenplot(data=dados, x="categoria", y="receita", ax=ax)
st.pyplot(fig, width="stretch")
```

## 10. Aplicação completa

O arquivo `streamlit_demo/app.py` contém:

- menu lateral com quatro páginas;
- filtros de categorias, região e meses;
- indicadores e tabela expansível;
- três abas de gráficos Altair, incluindo seleção pela legenda;
- abas com treemap e dispersão 3D em Plotly;
- abas com boxenplot e swarmplot em Seaborn.

Execute com:

```bash
streamlit run notebooks/streamlit_demo/app.py
```

## Boas práticas

- Separe carregamento, transformação e apresentação dos dados.
- Use `@st.cache_data` em leituras e cálculos determinísticos.
- Comece com poucos filtros e uma tarefa clara por página.
- Informe unidades, fontes e recortes dos dados.
- Teste o comportamento quando um filtro produz uma tabela vazia.
- Evite gráficos apenas porque são interativos; escolha-os de acordo com a pergunta.

### Referências

- [Documentação do Streamlit](https://docs.streamlit.io/)
- [Gráficos Altair no Streamlit](https://docs.streamlit.io/develop/api-reference/charts/st.altair_chart)
- [Layouts e contêineres](https://docs.streamlit.io/develop/api-reference/layout)